## Import Needed Libraries

In [1]:
import model
import data
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torch.optim as optim
import time
import matplotlib.pyplot as plt
import scipy.io as sio


Inputs: torch.Size([800, 8, 11993])
After First Convolution + ReLU: torch.Size([800, 32, 11993])
After First Max Pooling: torch.Size([800, 32, 5996])
After Second Convolution + ReLU: torch.Size([800, 128, 5996])
After Second Max Pooling: torch.Size([800, 128, 2998])
After Third Convolution + ReLU: torch.Size([800, 256, 2998])
After Third Max Pooling: torch.Size([800, 256, 1499])
After Flattening: torch.Size([800, 383744])
After Full Connection 1 + ReLU: torch.Size([800, 1500])
After Full Connection 2: torch.Size([800, 256])
After Full Connection 3: torch.Size([800, 6])


## Load Dataset

In [2]:
train_dataset = data.InverseData(mat_file="../Data/NEDC_lite.mat", outputs_npy="../Data/outputs_train.npy", ks_npy="../Data/ks_train.npy")
val_dataset = data.InverseData(mat_file="../Data/NEDC_lite.mat", outputs_npy="../Data/outputs_val.npy", ks_npy="../Data/ks_val.npy")
test_dataset = data.InverseData(mat_file="../Data/NEDC_lite.mat", outputs_npy="../Data/outputs_test.npy", ks_npy="../Data/ks_test.npy")

train_modelouts = np.load("../Data/outputs_train.npy").astype(np.float32)
val_modelouts   = np.load("../Data/outputs_val.npy").astype(np.float32)
test_modelouts  = np.load("../Data/outputs_test.npy").astype(np.float32)

train_A = np.load("../Data/ks_train.npy").astype(np.float64)
val_A = np.load("../Data/ks_val.npy").astype(np.float64)
test_A = np.load("../Data/ks_test.npy").astype(np.float64)

In [3]:
# Check loaded file sizes

print('Training Model Outputs Array Size:', train_modelouts.shape)
print('Validation Model Outputs Array Size:', val_modelouts.shape)
print('Test Model Outputs Array Size:', test_modelouts.shape)

print('Training As Array Size:', train_A.shape)
print('Validation As Array Size:', val_A.shape)
print('Test As Array Size:', test_A.shape)


Training Model Outputs Array Size: (800, 11993)
Validation Model Outputs Array Size: (100, 11993)
Test Model Outputs Array Size: (100, 11993)
Training As Array Size: (800, 6)
Validation As Array Size: (100, 6)
Test As Array Size: (100, 6)


In [4]:
mat_path = "../Data/NEDC_lite.mat"

mat = sio.loadmat(mat_path, squeeze_me = True, struct_as_record = False)

def get_signal(name):
    return np.asarray(mat[name].signals.values, dtype=np.float32).reshape(-1)

F_NOx = get_signal("F_NOx_sensor") # upstream NOx
Dosing = get_signal("Dosing") # dosing signal
Temp = get_signal("Temp") # exhaust temp
ExhaustFlow = get_signal("ExhaustFlow") # exhaust flow
Adblue = get_signal("adblue_mg") # Adblue signal
O2 = get_signal("O2") # O2
Temp_DOC_up = get_signal("Temp_DOC_up") # temp upstream of DOC

# Common signals for all simulations
# Shape: [time, 5]

simmodel_signals = np.stack([F_NOx, Dosing, Temp, ExhaustFlow, Adblue, O2, Temp_DOC_up], axis=1)

print("Simulink Model Signals:", simmodel_signals.shape)

Simulink Model Signals: (11993, 7)


## Dataset Preparation

In [5]:
class CustomDataset(Dataset):
  def __init__(self, train_set, val_set, test_set):
    
    self.train_set = torch.tensor(train_set).to(torch.float32)
    self.val_set = torch.tensor(val_set).to(torch.float32)
    self.test_set = torch.tensor(test_set).to(torch.float32)

  def __len__(self):

    return len(self.train_set)

  def __getitem__(self, idx):

    return self.train_set[idx], self.val_set[idx], self.test_set[idx]


In [6]:
# reshape inputs into 3D arrays

N_train = len(train_modelouts)
N_val = len(val_modelouts)
N_test = len(test_modelouts)

timesteps_train = train_modelouts.shape[-1]
timesteps_val = val_modelouts.shape[-1]
timesteps_test = test_modelouts.shape[-1]

A_set_size = train_A.shape[-1]

train_modelouts_size = [N_train, timesteps_train]
val_modelouts_size = [N_val, timesteps_val]
test_modelouts_size = [N_test, timesteps_test]

train_A_size = [N_train, A_set_size]
val_A_size = [N_val, A_set_size]
test_A_size = [N_test, A_set_size]

train_modelouts_inputs = np.expand_dims(np.reshape(train_modelouts, train_modelouts_size), axis=1)
val_modelouts_inputs = np.expand_dims(np.reshape(val_modelouts, val_modelouts_size), axis = 1)
test_modelouts_inputs = np.expand_dims(np.reshape(test_modelouts, test_modelouts_size), axis=1)

train_A_inputs = np.expand_dims(np.reshape(train_A, train_A_size), axis=1)
val_A_inputs = np.expand_dims(np.reshape(val_A, val_A_size), axis = 1)
test_A_inputs = np.expand_dims(np.reshape(test_A, test_A_size), axis=1)

# clear memory

#del train_x
#del test_x

print(train_modelouts_inputs.shape, val_modelouts_inputs.shape, test_modelouts_inputs.shape)
print(train_A_inputs.shape, val_A_inputs.shape, test_A_inputs.shape)

# Loading the data

train_modelouts_loader = DataLoader(train_modelouts_inputs, batch_size = 100, shuffle = True)
val_modelouts_loader = DataLoader(val_modelouts_inputs, batch_size = 10, shuffle = True)
test_modelouts_loader = DataLoader(test_modelouts_inputs, batch_size = 10, shuffle = True)

train_A_loader = DataLoader(train_A_inputs, batch_size = 100, shuffle = True)
val_A_loader = DataLoader(val_A_inputs, batch_size = 10, shuffle = True)
test_A_loader = DataLoader(test_A_inputs, batch_size = 10, shuffle = True)

# Calculate and display the number of mini-batches

num_train_modelouts_minibatches = len(train_modelouts_loader)
num_val_modelouts_minibatches = len(val_modelouts_loader)
num_test_modelouts_minibatches = len(test_modelouts_loader)

num_train_A_minibatches = len(train_A_loader)
num_val_A_minibatches = len(val_A_loader)
num_test_A_minibatches = len(test_A_loader)

print(f"Number of Mini-Batches: {num_train_modelouts_minibatches}")
print(f"Number of Mini-Batches: {num_val_modelouts_minibatches}")
print(f"Number of Mini-Batches: {num_test_modelouts_minibatches}")
print(f"Number of Mini-Batches: {num_train_A_minibatches}")
print(f"Number of Mini-Batches: {num_val_A_minibatches}")
print(f"Number of Mini-Batches: {num_test_A_minibatches}")

(800, 1, 11993) (100, 1, 11993) (100, 1, 11993)
(800, 1, 6) (100, 1, 6) (100, 1, 6)
Number of Mini-Batches: 8
Number of Mini-Batches: 10
Number of Mini-Batches: 10
Number of Mini-Batches: 8
Number of Mini-Batches: 10
Number of Mini-Batches: 10


## Define Hyper/parameters

In [7]:
# Setup for a parametric study

size_kernel = [3, 5, 10, 15] # kernel size, set params_user to 0
size_batch = [16, 32, 64, 100] # batch size, set params_user to 1
reg_param = [1e-2, 1e-3, 1e-4] # regularization parameter, set params_user to 2

params_user = 2 # choose between 0, 1, 2

N_epochs = 500 # can vary

# Define loss criteria

criterion = nn.MSELoss()

# total runtime

time_total = time.time()


## Model Instantiation

In [ ]:
#model = model.CNN(channels_in = 1)

## Training Loop

In [8]:
# Parameter study case: select which parameter to vary

if params_user == 0:
    params = size_kernel
elif params_user == 1:
    params = size_batch
elif params_user == 2:
    params = reg_param

train_loss_metrics = []
test_loss_metrics = []
train_accuracy_metrics = []
test_accuracy_metrics = []

param_time = []

for param in params:

    if params_user == 0:
        k = param
        N_batch = 100
        reg_param = 1e-3
    elif params_user == 1:
        N_batch = param
        k = 3
        reg_param = 1e-3
    elif params_user == 2:
        reg_param = param
        k = 3
        N_batch = 100

    # Per epoch
    
    train_loss_history = []
    test_loss_history = []
    train_accuracy_history = []
    test_accuracy_history = []

    t_0 = time.time()

    t_0_main = time.time()

    # initialize model

    model = model.CNN(channels_in = 1)

    # Define optimizer. Use ADAM with regularization parameter as weight decay
    
    optimizer = optim.Adam(model.parameters(), weight_decay = reg_param)

    for epoch in range(1, N_epochs + 1):
    #for epoch in range(2):
        for batch in train_modelouts_loader:
            x_batch = batch[0]
            y_batch = batch[1]

            # zero the gradients
            
            optimizer.zero_grad()
            
            # forward
            
            y_pred = model(x_batch)

            # compute loss
            
            loss = criterion(y_pred[:,0], y_batch[:,0])
            
            # back propagation
            
            loss.backward()
            
            # update the parameters theta
            
            optimizer.step()

            #acc = torch.abs(y_pred - y_batch)/y_batch
            #acc[~torch.isfinite(acc)] = 0
            #accuracy = torch.nanmean(acc)

            err = y_pred - y_batch
            error = err.cpu().detach().numpy()
            y_b = y_batch.cpu().detach().numpy()
            accuracy = np.linalg.norm(error) / np.linalg.norm(y_b)
            
            # removed .to("cpu").detach().numpy() for now
            train_accuracy_history.append(accuracy)
            train_loss_history.append(loss.item())

            with torch.no_grad():
                
                model.eval()

                y_test_pred = model(torch.Tensor(test_inputs).to(torch.float32).to("cuda"))
                
                # loss is calculated using viscous units
                
                test_loss_mean = criterion(y_test_pred[:,0]/utau_test, torch.Tensor(test_labels[:,0]/utau_train).to(torch.float32).to('cuda'))
                test_loss_rms = criterion(y_test_pred[:,1]/utau_test, torch.Tensor(test_labels[:,1]/utau_train).to(torch.float32).to('cuda'))
                test_loss = test_loss_mean + lambda_weight * test_loss_rms
                test_loss_history.append(test_loss.item())

                test_acc = np.abs(y_test_pred.to("cpu").detach().numpy() - test_labels)/test_labels
                test_acc[~np.isfinite(test_acc)] = 0
                test_accuracy = np.nanmean(test_acc)
                test_accuracy_history.append(test_accuracy)

            model.train()

        if epoch % int(N_epochs/5) == 0: # asumming N_epochs/20 is an integer
            print(f"Epoch [{epoch}/{int(N_epochs)}],")
            print(f"Data: Loss: {loss.item()} Accuracy: {accuracy}")

    train_loss_metrics.append(np.mean(train_loss_history[-1000:]))
    test_loss_metrics.append(np.mean(test_loss_history[-1000:]))
    train_accuracy_metrics.append(np.mean(train_accuracy_history[-1000:]))
    test_accuracy_metrics.append(np.mean(test_accuracy_history[-1000:]))
    
    plt.figure(figsize = (8,6))
    plt.box(True)
    plt.plot(train_loss_history, label = "Training Loss")
    plt.plot(test_loss_history, label = "Test Loss")
    plt.gca().set_yscale('log')
    plt.legend()
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title(f'Loss vs. Epoch for Lambda = {param}')
    plt.show()

    final_train_loss = train_loss_history[-1]
    final_test_loss = test_loss_history[-1]

    print(f"Final Losses for Lambda = {param}: Training Loss - {final_train_loss}, Test Loss - {final_test_loss}")
    
    t_1_main = time.time()

    print(f"Runtime {t_1_main-t_0_main} s.")
    param_time.append(t_1_main-t_0_main)
    

RuntimeError: mat1 and mat2 shapes cannot be multiplied (256x1499 and 383744x1500)